In [1]:
!pip install kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"tanushakatakam","key":"4d681e49ee2e638d6a6cea0b418d3d8a"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download lakshmi25npathi/bike-sharing-dataset

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/bike-sharing-dataset
License(s): unknown
100% 286k/286k [00:00<00:00, 105MB/s]



In [5]:
!unzip -o bike-sharing-dataset.zip -d bike-sharing-dataset

Archive:  bike-sharing-dataset.zip
  inflating: bike-sharing-dataset/Readme.txt  
  inflating: bike-sharing-dataset/day.csv  
  inflating: bike-sharing-dataset/hour.csv  


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

from prophet import Prophet

In [7]:
df = pd.read_csv("bike-sharing-dataset/hour.csv")
df.head(10)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,0,1,1
5,6,2011-01-01,1,0,1,5,0,6,0,2,0.24,0.2576,0.75,0.0896,0,1,1
6,7,2011-01-01,1,0,1,6,0,6,0,1,0.22,0.2727,0.80,0.0000,2,0,2
7,8,2011-01-01,1,0,1,7,0,6,0,1,0.20,0.2576,0.86,0.0000,1,2,3
8,9,2011-01-01,1,0,1,8,0,6,0,1,0.24,0.2879,0.75,0.0000,1,7,8
9,10,2011-01-01,1,0,1,9,0,6,0,1,0.32,0.3485,0.76,0.0000,8,6,14


# **Data** **Preprocessing**

In [8]:
df['dteday'] = pd.to_datetime(df['dteday'])

In [9]:
df.rename(columns={'dteday':'datetime', 'cnt':'count'}, inplace=True)

In [10]:
df.drop(['instant', 'casual', 'registered'], axis=1, inplace=True)

# **Feature** **Engineering**

In [11]:
df['hour'] = df['hr']
df['day'] = df['datetime'].dt.day
df['month'] = df['datetime'].dt.month
df['weekday'] = df['datetime'].dt.weekday

# **Lag** **Feature**

In [12]:
df = df.sort_values('datetime')
df['lag_1'] = df['count'].shift(1)
df = df.dropna()

In [13]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 17378 entries, 23 to 17378
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   datetime    17378 non-null  datetime64[ns]
 1   season      17378 non-null  int64         
 2   yr          17378 non-null  int64         
 3   mnth        17378 non-null  int64         
 4   hr          17378 non-null  int64         
 5   holiday     17378 non-null  int64         
 6   weekday     17378 non-null  int32         
 7   workingday  17378 non-null  int64         
 8   weathersit  17378 non-null  int64         
 9   temp        17378 non-null  float64       
 10  atemp       17378 non-null  float64       
 11  hum         17378 non-null  float64       
 12  windspeed   17378 non-null  float64       
 13  count       17378 non-null  int64         
 14  hour        17378 non-null  int64         
 15  day         17378 non-null  int32         
 16  month       17378 non-null

,datetime,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,count,hour,day,month,lag_1
count,17378,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000,17378.000000
mean,2012-01-02 04:38:55.090343936,2.501726,0.502589,6.538094,11.547416,0.028772,3.011336,0.682760,1.425308,0.497002,0.475786,0.627218,0.190109,189.473069,11.547416,15.684256,6.538094,189.471170
min,2011-01-01 00:00:00,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.020000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000
25%,2011-07-04 06:00:00,2.000000,0.000000,4.000000,6.000000,0.000000,1.000000,0.000000,1.000000,0.340000,0.333300,0.480000,0.104500,40.000000,6.000000,8.000000,4.000000,40.000000
50%,2012-01-02 00:00:00,3.000000,1.000000,7.000000,12.000000,0.000000,3.000000,1.000000,1.000000,0.500000,0.484800,0.630000,0.194000,142.000000,12.000000,16.000000,7.000000,142.000000
75%,2012-07-02 00:00:00,3.000000,1.000000,10.000000,18.000000,0.000000,5.000000,1.000000,2.000000,0.660000,0.621200,0.780000,0.253700,281.000000,18.000000,23.000000,10.000000,281.000000
max,2012-12-31 00:00:00,4.000000,1.000000,12.000000,23.000000,1.000000,6.000000,1.000000,4.000000,1.000000,1.000000,1.000000,0.850700,977.000000,23.000000,31.000000,12.000000,977.000000
std,NaN,1.106891,0.500008,3.438618,6.914049,0.167170,2.001967,0.465415,0.639367,0.192552,0.171849,0.192930,0.122335,181.388045,6.914049,8.788921,3.438618,181.389688


In [14]:
X = df.drop(['count', 'datetime'], axis=1)
y = df['count']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# **Model** **1**: **Random** **Forest**



In [16]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [17]:
y_pred = rf.predict(X_test)

In [18]:
print("Random Forest Performance:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

Random Forest Performance:
MAE: 27.553437859608746
R2 Score: 0.943369594518686


In [19]:
df['RF_Predicted_Demand'] = rf.predict(X)

# **Model 2: Prophet(Time Series)**

In [20]:
ts_df = df[['datetime', 'count']].copy()
ts_df.columns = ['ds', 'y']

In [21]:
prophet_model = Prophet()
prophet_model.fit(ts_df)

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


In [22]:
future = prophet_model.make_future_dataframe(periods=48, freq='H')
forecast = prophet_model.predict(future)

/usr/local/lib/python3.12/dist-packages/prophet/forecaster.py:1875: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(


In [23]:
prophet_output = forecast[['ds', 'yhat']]

In [24]:
df = df.merge(prophet_output, left_on='datetime', right_on='ds', how='left')
df.rename(columns={'yhat': 'Prophet_Predicted_Demand'}, inplace=True)
df.drop('ds', axis=1, inplace=True)

# **Model Comparison**

In [25]:
comparison_df = df[['count', 'RF_Predicted_Demand', 'Prophet_Predicted_Demand']].dropna()

rf_r2 = r2_score(comparison_df['count'], comparison_df['RF_Predicted_Demand'])
prophet_r2 = r2_score(comparison_df['count'], comparison_df['Prophet_Predicted_Demand'])

print("\nModel Comparison:")
print("Random Forest R2:", rf_r2)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("Prophet R2:", prophet_r2)
print("Prophet MAE:", mean_absolute_error(comparison_df['count'], comparison_df['Prophet_Predicted_Demand']))


Model Comparison:
Random Forest R2: 0.9812682606581251
MAE: 27.553437859608746
Prophet R2: 0.14734449955551565
Prophet MAE: 128.23861817607903


#**Dataset** **with** **added** **predictions**

In [26]:
df.to_csv("bike_demand_dashboard.csv", index=False)

In [27]:
df.columns

Index(['datetime', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'count', 'hour', 'day', 'month', 'lag_1', 'RF_Predicted_Demand',
       'Prophet_Predicted_Demand'],
      dtype='object')

In [28]:
from google.colab import files
files.download("bike_demand_dashboard.csv")